In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import ast

mlpath = kagglehub.dataset_download("sherinclaudia/movielens")

print("Path to dataset files:", mlpath)

krpath = kagglehub.dataset_download("arashnic/kuairec-recommendation-system-data-density-100")

print("Path to dataset files:", krpath)

Using Colab cache for faster access to the 'movielens' dataset.
Path to dataset files: /kaggle/input/movielens
Using Colab cache for faster access to the 'kuairec-recommendation-system-data-density-100' dataset.
Path to dataset files: /kaggle/input/kuairec-recommendation-system-data-density-100


In [ ]:
import pandas as pd
kr_bigm=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/big_matrix.csv")
print(kr_bigm.head(5))

   user_id  video_id  play_duration  video_duration                     time  \
0        0      3649          13838           10867  2020-07-05 00:08:23.438   
1        0      9598          13665           10984  2020-07-05 00:13:41.297   
2        0      5262            851            7908  2020-07-05 00:16:06.687   
3        0      1963            862            9590  2020-07-05 00:20:26.792   
4        0      8234            858           11000  2020-07-05 00:43:05.128   

       date     timestamp  watch_ratio  
0  20200705  1.593879e+09     1.273397  
1  20200705  1.593879e+09     1.244082  
2  20200705  1.593879e+09     0.107613  
3  20200705  1.593880e+09     0.089885  
4  20200705  1.593881e+09     0.078000  


In [ ]:
kr_smallm=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/small_matrix.csv")
print(kr_smallm.head(5))

   user_id  video_id  play_duration  video_duration                     time  \
0       14       148           4381            6067  2020-07-05 05:27:48.378   
1       14       183          11635            6100  2020-07-05 05:28:00.057   
2       14      3649          22422           10867  2020-07-05 05:29:09.479   
3       14      5262           4479            7908  2020-07-05 05:30:43.285   
4       14      8234           4602           11000  2020-07-05 05:35:43.459   

         date     timestamp  watch_ratio  
0  20200705.0  1.593898e+09     0.722103  
1  20200705.0  1.593898e+09     1.907377  
2  20200705.0  1.593898e+09     2.063311  
3  20200705.0  1.593898e+09     0.566388  
4  20200705.0  1.593899e+09     0.418364  


In [ ]:
kr_cats=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/item_categories.csv")
print(kr_cats.head(5))

   video_id     feat
0         0      [8]
1         1  [27, 9]
2         2      [9]
3         3     [26]
4         4      [5]


In [ ]:
ml_movies=pd.read_csv("/kaggle/input/movielens/movies.dat", sep='::', names=['MovieID', 'Title', 'Genres'], engine='python', encoding='latin-1')
# Fill NaN values with empty strings before splitting, then filter out empty strings from lists
ml_movies['Genres'] = ml_movies['Genres'].fillna('').str.split('|').apply(lambda x: [g for g in x if g])
print(ml_movies.head(5))

   MovieID                               Title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                             Genres  
0   [Animation, Children's, Comedy]  
1  [Adventure, Children's, Fantasy]  
2                 [Comedy, Romance]  
3                   [Comedy, Drama]  
4                          [Comedy]  


In [ ]:
ml_ratings=pd.read_csv("/kaggle/input/movielens/ratings.dat", sep='::', names=["UID", "MovieID", "Rating", "Timestamp"], engine='python', encoding='latin-1')
print(ml_ratings.head(5))

   UID  MovieID  Rating  Timestamp
0    1     1193       5  978300760
1    1      661       3  978302109
2    1      914       3  978301968
3    1     3408       4  978300275
4    1     2355       5  978824291


In [ ]:
ml_users=pd.read_csv("/kaggle/input/movielens/users.dat", sep='::', names=["UID", "SEX", "AGE", "OCC", "PIN"], engine='python', encoding='latin-1')
print(ml_users.head(5))

   UID SEX  AGE  OCC    PIN
0    1   F    1   10  48067
1    2   M   56   16  70072
2    3   M   25   15  55117
3    4   M   45    7  02460
4    5   M   25   20  55455


In [ ]:
assert set(kr_smallm.user_id).issubset(set(kr_bigm.user_id))
assert set(kr_smallm.video_id).issubset(set(kr_bigm.video_id))

In [ ]:
print(ml_movies.shape)
print(ml_ratings.shape)
print(ml_users.shape)

(3883, 3)
(1000209, 4)
(6040, 5)


In [ ]:
assert set(kr_smallm.user_id).issubset(set(kr_bigm.user_id))
assert set(kr_smallm.video_id).issubset(set(kr_bigm.video_id))
print("KuaiRec: small_matrix fully covered by big_matrix vocabulary OK")

KuaiRec: small_matrix fully covered by big_matrix vocabulary OK


In [ ]:
KR_THRESHOLD = 2.0
ML_THRESHOLD = 3.5

kr_bigm['label'] = (kr_bigm['watch_ratio'] > KR_THRESHOLD).astype(int)
kr_smallm['label'] = (kr_smallm['watch_ratio'] > KR_THRESHOLD).astype(int)
print("KuaiRec positive rates:", kr_bigm['label'].mean(), kr_smallm['label'].mean())

ml_ratings['label'] = (ml_ratings['Rating'] > ML_THRESHOLD).astype(int)
print("MovieLens positive rate:", ml_ratings['label'].mean())

KuaiRec positive rates: 0.07472703671256263 0.04643894991414648
MovieLens positive rate: 0.5751607913945985


In [ ]:
kr_user2idx = {u: i for i, u in enumerate(sorted(kr_bigm.user_id.unique()))}
kr_item2idx = {v: i for i, v in enumerate(sorted(kr_bigm.video_id.unique()))}

for df in (kr_bigm, kr_smallm):
    df['user_idx'] = df['user_id'].map(kr_user2idx)
    df['item_idx'] = df['video_id'].map(kr_item2idx)

n_kr_items = len(kr_item2idx)

ml_user2idx = {u: i for i, u in enumerate(sorted(ml_ratings.UID.unique()))}
ml_item2idx = {m: i for i, m in enumerate(sorted(ml_ratings.MovieID.unique()))}

ml_ratings['user_idx'] = ml_ratings['UID'].map(ml_user2idx)
ml_ratings['item_idx'] = ml_ratings['MovieID'].map(ml_item2idx)

n_ml_items = len(ml_item2idx)

In [ ]:
from sklearn.model_selection import train_test_split

ml_train, ml_test = train_test_split(
    ml_ratings, test_size=0.2, random_state=42, stratify=ml_ratings['label']
)
print(f"MovieLens: {len(ml_train)} train / {len(ml_test)} test interactions")

MovieLens: 800167 train / 200042 test interactions


In [ ]:
def make_recbole_dirs(base_path, dataset_names):
    for name in dataset_names:
        os.makedirs(os.path.join(base_path, name), exist_ok=True)

def write_inter_file(df, path, user_col='user_idx', item_col='item_idx', label_col='label'):
    out = df[[user_col, item_col, label_col]].copy()
    out.columns = ['user_id:token', 'item_id:token', 'label:float']
    out.to_csv(path, sep='\t', index=False)

def write_item_file_multivalued(item_ids, feat_lists, path, feat_field='genre'):
    """feat_lists: list-of-lists aligned with item_ids (raw category values, not multi-hot)."""
    rows = []
    for iid, feats in zip(item_ids, feat_lists):
        feat_str = ' '.join(str(f) for f in feats) if len(feats) > 0 else ''
        rows.append((iid, feat_str))
    out = pd.DataFrame(rows, columns=['item_id:token', f'{feat_field}:token_seq'])
    out.to_csv(path, sep='\t', index=False)


BASE = '/content/recbole_data'
make_recbole_dirs(BASE, ['ml-1m', 'kuairec'])

# ---- MovieLens: carve valid OUT OF TRAIN, never touch test ----
ml_valid = ml_train.sample(frac=0.1, random_state=42)
ml_train_final = ml_train.drop(ml_valid.index)

write_inter_file(ml_train_final, f'{BASE}/ml-1m/ml-1m.train.inter')
write_inter_file(ml_valid,       f'{BASE}/ml-1m/ml-1m.valid.inter')
write_inter_file(ml_test,        f'{BASE}/ml-1m/ml-1m.test.inter')

write_item_file_multivalued(
    ml_movies['MovieID'].map(ml_item2idx).dropna().astype(int).tolist(),
    ml_movies.loc[ml_movies['MovieID'].map(ml_item2idx).notna(), 'Genres'].tolist(),
    f'{BASE}/ml-1m/ml-1m.item',
    feat_field='genre'
)

n_kept = ml_movies['MovieID'].map(ml_item2idx).notna().sum()
print(f"MovieLens items kept in .item file: {n_kept}/{len(ml_movies)}")

# ---- KuaiRec: train=big_matrix, test=small_matrix (per dataset's own protocol) ----
kr_valid = kr_bigm.sample(frac=0.02, random_state=42)   # small slice of big for early stopping
kr_train_final = kr_bigm.drop(kr_valid.index)

write_inter_file(kr_train_final, f'{BASE}/kuairec/kuairec.train.inter')
write_inter_file(kr_valid,       f'{BASE}/kuairec/kuairec.valid.inter')
write_inter_file(kr_smallm,      f'{BASE}/kuairec/kuairec.test.inter')

kr_cats['feat'] = kr_cats['feat'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

kr_items_mask = kr_cats['video_id'].map(kr_item2idx).notna()
write_item_file_multivalued(
    kr_cats.loc[kr_items_mask, 'video_id'].map(kr_item2idx).astype(int).tolist(),
    kr_cats.loc[kr_items_mask, 'feat'].tolist(),
    f'{BASE}/kuairec/kuairec.item',
    feat_field='feat'
)

n_kr_kept = kr_items_mask.sum()
print(f"KuaiRec items kept in .item file: {n_kr_kept}/{len(kr_cats)}")

print("Atomic files written.")

# ---- Quick peek to confirm token_seq columns are populated ----
print(pd.read_csv(f'{BASE}/ml-1m/ml-1m.item', sep='\t').head())
print(pd.read_csv(f'{BASE}/kuairec/kuairec.item', sep='\t').head())

MovieLens items kept in .item file: 3706/3883
KuaiRec items kept in .item file: 10728/10728
Atomic files written.
   item_id:token               genre:token_seq
0              0   Animation Children's Comedy
1              1  Adventure Children's Fantasy
2              2                Comedy Romance
3              3                  Comedy Drama
4              4                        Comedy
   item_id:token feat:token_seq
0              0              8
1              1           27 9
2              2              9
3              3             26
4              4              5


In [ ]:
# =========================================================
# Interaction-level leakage check: big_matrix (train) vs small_matrix (eval)
# =========================================================

# Positive pairs only — the (user, item) pairs a model could actually "memorize"
kr_train_pos_pairs = set(
    zip(kr_train_final.loc[kr_train_final['label'] == 1, 'user_idx'],
        kr_train_final.loc[kr_train_final['label'] == 1, 'item_idx'])
)

kr_eval_pos_pairs = set(
    zip(kr_smallm.loc[kr_smallm['label'] == 1, 'user_idx'],
        kr_smallm.loc[kr_smallm['label'] == 1, 'item_idx'])
)

overlap_pairs = kr_train_pos_pairs & kr_eval_pos_pairs
print(f"KuaiRec: {len(overlap_pairs)} (user, item) positive pairs appear in BOTH "
      f"train and eval out of {len(kr_eval_pos_pairs)} eval positives "
      f"({100 * len(overlap_pairs) / max(len(kr_eval_pos_pairs), 1):.2f}%)")

# Mask overlapping pairs OUT of the eval set BEFORE training/evaluation —
# per KuaiRec's protocol, eval should measure generalization to interactions
# the model hasn't already seen labeled as positive in training.
kr_smallm['pair'] = list(zip(kr_smallm['user_idx'], kr_smallm['item_idx']))
kr_smallm_clean = kr_smallm[~kr_smallm['pair'].isin(overlap_pairs)].drop(columns=['pair']).reset_index(drop=True)

print(f"KuaiRec eval set: {len(kr_smallm)} rows -> {len(kr_smallm_clean)} rows after masking overlap")

# From this point on, use kr_smallm_clean as the eval/test set everywhere
# (RecBole .inter file, scorer input, etc.) instead of kr_smallm.
write_inter_file(kr_smallm_clean, f'{BASE}/kuairec/kuairec.test.inter')

KuaiRec: 0 (user, item) positive pairs appear in BOTH train and eval out of 217175 eval positives (0.00%)
KuaiRec eval set: 4676570 rows -> 4676570 rows after masking overlap


In [ ]:
def compute_recall_ndcg_at_k(user_topk: dict, user_ground_truth: dict, k=20):
    """
    user_topk: {user_id: [item_id, item_id, ...]} ranked descending by score,
               already filtered to exclude train-seen items.
    user_ground_truth: {user_id: set(item_id)} — eval-set positives.
    Every arm (LightGCN, DeepFM, Wide&Deep, SVD) must convert its raw
    predictions into this shape before being scored here.
    """
    recalls, ndcgs = [], []
    for user, ranked_items in user_topk.items():
        gt = user_ground_truth.get(user, set())
        if not gt:
            continue
        ranked_k = ranked_items[:k]
        hits = np.array([1 if item in gt else 0 for item in ranked_k])

        recall = hits.sum() / min(len(gt), k)
        recalls.append(recall)

        dcg = np.sum(hits / np.log2(np.arange(2, k + 2)))
        ideal_hits = min(len(gt), k)
        idcg = np.sum(1.0 / np.log2(np.arange(2, ideal_hits + 2))) if ideal_hits > 0 else 0.0
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {f'Recall@{k}': np.mean(recalls), f'NDCG@{k}': np.mean(ndcgs)}